# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, inspect, and analyze the [FAIR2 Mass Spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library and Python tools.

### Dataset Source
The dataset is described by a Croissant schema accessible at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed. Uncomment if running in a fresh environment.
!pip install mlcroissant

## 1. Data Loading
Load both metadata and data records using `mlcroissant`. We'll identify available record sets and work with their identifiers (`@id`).

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Load metadata as a dict
metadata_dict = dataset.metadata.to_json()
print(f"Dataset Title: {getattr(dataset.metadata, 'name', 'Unknown')}")
print(f"Description: {getattr(dataset.metadata, 'description', 'No description.')}")
print(f"Record sets: {getattr(dataset.metadata, 'recordSet', [])}")

## 2. Data Overview
Let's enumerate the available record sets, their fields, and `@id` identifiers for transparent exploration.

In [ ]:
from pprint import pprint

# List all available record sets and fields
record_sets = getattr(dataset.metadata, 'recordSet', [])

if not record_sets:
    # Sometimes the recordSet list may need to be retrieved from hasPart
    record_sets = getattr(dataset.metadata, 'hasPart', [])
print(f"Record sets (@id):\n{record_sets}\n")

record_set_fields = {}
for record_set_id in record_sets:
    try:
        record_set = dataset.metadata.entity(record_set_id)
        fields = getattr(record_set, 'field', [])
        print(f"Record Set: {record_set_id}")
        print(f"\tName: {getattr(record_set, 'name', '')}")
        print(f"\tDescription: {getattr(record_set, 'description', '')}")
        print("\tFields:")
        for f_id in fields:
            f_entity = dataset.metadata.entity(f_id)
            print(f"\t - {f_id} (name: {getattr(f_entity, 'name', '')}, type: {getattr(f_entity, 'dataType', '')})")
        print()
        record_set_fields[record_set_id] = fields
    except Exception as e:
        print(f"Could not access record set {record_set_id}: {str(e)}\n")

## 3. Data Extraction
We will load one or more of the available record sets into pandas DataFrames. All field and record set references use their Croissant `@id` values for full reproducibility. We'll use the first available record set for further analysis.

In [ ]:
# Choose all found record sets (by @id)
extracted_record_sets = record_sets
dataframes = {}

for rs_id in extracted_record_sets:
    print(f"\nExtracting records for record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"  Fields: {list(df.columns)}")
    print(f"  Number of records: {len(df)}")
    print(df.head(2))

# We will use the first available record set for further steps
if extracted_record_sets:
    active_record_set_id = extracted_record_sets[0]
    df = dataframes[active_record_set_id]
    print(f"\nWorking with: {active_record_set_id}")
    print(f"Columns: {df.columns.tolist()}")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Let's process and analyze the data. We will select a numeric field (by `@id` as listed previously), filter, normalize, and group the data. Adjust field IDs as needed, based on the actual dataset fields discovered above.

In [ ]:
# Pick a numeric field (update as needed based on available columns)
# Example: Find a field that looks numeric from df.columns
import numpy as np

numeric_field = None
for col in df.columns:
    # Heuristic: If column datatype is numeric, or has numeric name
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field = col
        break
if not numeric_field:
    # Try columns that look like counts or MW
    for col in df.columns:
        if any([k in col.lower() for k in ['count', 'abundance', 'mw', 'coverage', 'pi']]):
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
                if df[col].notna().sum() > 0:
                    numeric_field = col
                    break
            except:
                pass
if not numeric_field:
    print("No suitable numeric field found for analysis.")
else:
    print(f"Using numeric field: {numeric_field}")
    # Remove NaNs
    filtered_df = df[df[numeric_field].notnull()].copy()
    # Filter using 10th percentile as threshold
    threshold = filtered_df[numeric_field].quantile(0.10)
    filtered_df = filtered_df[filtered_df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold} (10th percentile): {filtered_df.shape[0]} records")

    # Normalize the numeric field (z-score)
    normalized = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    filtered_df[f"{numeric_field}_normalized"] = normalized
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a categorical field
    group_field = None
    for col in df.columns:
        if col != numeric_field and df[col].dtype == object and df[col].nunique() < 10:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].agg(['mean', 'std', 'count'])
        print(f"Grouped mean/std/count by {group_field}:")
        print(grouped_df.head())
    else:
        print("No suitable categorical field found for grouping.")

## 5. Visualization
Plot the distribution of the selected numeric field and a grouped bar chart if grouping was possible. This can help identify outliers, data quality, and groupwise differences.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if numeric_field:
    # Distribution
    sns.histplot(filtered_df[numeric_field], bins=30, kde=True, ax=axes[0], color='steelblue')
    axes[0].set_title(f"Distribution of {numeric_field}")
    axes[0].set_xlabel(numeric_field)

    # Grouped bar (if group_field)
    if 'group_field' in locals() and group_field:
        group_stats = (
            filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(numeric_field)
        )
        sns.barplot(data=group_stats, x=group_field, y=numeric_field, ax=axes[1], palette='muted')
        axes[1].set_title(f"Mean {numeric_field} by {group_field}")
        axes[1].set_xlabel(group_field)
        axes[1].set_ylabel(f"Mean {numeric_field}")
    else:
        axes[1].axis('off')
        axes[1].text(0.5, 0.5, 'No group field for grouping', ha='center', va='center')
plt.tight_layout()
plt.show()

## 6. Conclusion
This notebook demonstrated how to explore and process a Croissant-compliant FAIR dataset with `mlcroissant`. By referencing record sets and fields via their `@id`, you ensure repeatable, unambiguous code and make use of the enriched metadata structure for robust data science workflows.

**Key steps covered:**
- Load dataset and metadata using the Croissant schema URL
- Inspect and identify record sets and their field `@id`s
- Extract and examine tabular data via pandas DataFrames
- Filter, normalize, and group numeric data based on Croissant `@id` fields
- Visualize distributions and group patterns

Next steps could include feature engineering, advanced statistical modeling, or integration with additional biomedical datasets available in Croissant format.